# Classification: Perceptron (Single-Layer Neural Network)

## Justification of Preprocessing Strategy

### Strict Requirement for Standardization

The Perceptron is the foundational building block of artificial neural networks. It functions as a linear classifier that estimates a hyperplane boundary using a simple weight vector modification step.

Because it relies heavily on the dot product between weights and feature vectors, it is extremely sensitive to feature scales. Features with larger numerical ranges will dominate the weight updates, causing the algorithm to drift away from the true optimal separating boundary. To guarantee stable convergence and balanced feature contributions across our 100,000-sample dataset, standardization (StandardScaler) is strictly mandatory.

### The Linear Separation Limitation

The Perceptron updates its weights only when it makes a classification mistake. If the data is not perfectly linearly separable (which is highly likely in complex clinical data like diabetes diagnostics), the basic Perceptron algorithm will never fully converge, and its weights will oscillate indefinitely. To mitigate this, we introduce regularization penalties (L1/L2) during tuning to stabilize the weights and prevent overfitting on non-separable data.

## Experiment Design

We designed a tournament of 3 optimization levels using cross-validation to find the most robust Perceptron configuration for maximizing all the metrics:

- **Baseline**: Execute the Perceptron with pure Scikit-Learn default values to establish the baseline performance floor.
- **GridSearchCV**: Expand the worksheet loop into a cross-validated search over the learning rate (`eta0`) and regularization penalty types (`penalty`).
- **Optuna**: Use Bayesian optimization to explore a continuous and fine-grained range of the learning rate and alpha hyperparameter.

In [4]:
import pandas as pd
import numpy as np
import time
import mlflow
import optuna
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Perceptron
from sklearn.metrics import recall_score, accuracy_score, f1_score

# 1. MLflow Configuration
mlflow.set_tracking_uri("sqlite:///C:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente de Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/mlflow.db")
mlflow.set_experiment("Classification_Perceptron")

<Experiment: artifact_location=('file:c:/Users/Tiago Silva/Uni/OneDrive - Universidade Portucalense/Ambiente '
 'de '
 'Trabalho/Uni/3ano2sem/LAD/Grupo5_ProjetoLAD_Parte2/TrabalhoLAD/models/notebooks/Classification/NeuralNetworks/Perceptron/mlruns/13'), creation_time=1779120154794, experiment_id='13', last_update_time=1779120154794, lifecycle_stage='active', name='Classification_Perceptron', tags={}, workspace='default'>

In [5]:
df = pd.read_csv("C:\\Users\\Tiago Silva\\Uni\\OneDrive - Universidade Portucalense\\Ambiente de Trabalho\\Uni\\3ano2sem\\LAD\\Grupo5_ProjetoLAD_Parte2\\TrabalhoLAD\\data\\diabetes_dataset_new_variables.csv")
categorical_cols = ['gender', 'ethnicity', 'smoking_status', 'education_level', 'employment_status', 'age_groups', 'weight_status', 'income_level']
df_final = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_final.drop(["diagnosed_diabetes", "diabetes_stage"], axis=1)
y = df_final['diagnosed_diabetes']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns

# Scale features 
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

def log_metrics(y_true, y_pred, duration):
    """Utility function to log metrics to MLflow"""
    mlflow.log_metric("recall", recall_score(y_true, y_pred))
    mlflow.log_metric("accuracy", accuracy_score(y_true, y_pred))
    mlflow.log_metric("f1", f1_score(y_true, y_pred))
    mlflow.log_metric("fit_time", duration)

# ---------------------------------------------------------
# RUN 1: PERCEPTRON BASELINE (Strict Defaults)
# ---------------------------------------------------------
with mlflow.start_run(run_name="Perceptron_Baseline_Defaults"):
    # Initializing with strict Scikit-Learn defaults
    perceptron_base = Perceptron(random_state=42)
    
    start_time = time.time()
    perceptron_base.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    y_pred = perceptron_base.predict(X_test_scaled)
    
    mlflow.log_params(perceptron_base.get_params())
    mlflow.log_param("optimization", "none_default")
    log_metrics(y_test, y_pred, duration)

# ---------------------------------------------------------
# RUN 2: GRIDSEARCHCV 
# ---------------------------------------------------------
with mlflow.start_run(run_name="Perceptron_GridSearch"):
    param_grid = {
        'eta0': [0.0001, 0.001, 0.01, 0.1, 1.0],
        'penalty': ['l2', 'l1', None],
        'alpha': [0.0001, 0.001, 0.01]
    }
    
    grid = GridSearchCV(
        Perceptron(random_state=42, max_iter=1000),
        param_grid, cv=3, scoring='recall', n_jobs=-1
    )
    
    start_time = time.time()
    grid.fit(X_train_scaled, y_train)
    duration = time.time() - start_time
    
    best_perceptron = grid.best_estimator_
    y_pred_grid = best_perceptron.predict(X_test_scaled)
    
    mlflow.log_params(grid.best_params_)
    mlflow.log_param("optimization", "GridSearchCV")
    log_metrics(y_test, y_pred_grid, duration)

# ---------------------------------------------------------
# RUN 3: OPTUNA 
# ---------------------------------------------------------
def objective(trial):
    params = {
        "eta0": trial.suggest_float("eta0", 1e-5, 1.0, log=True),
        "penalty": trial.suggest_categorical("penalty", ["l2", "l1", None]),
        "alpha": trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
        "max_iter": trial.suggest_int("max_iter", 500, 2000),
        "random_state": 42
    }
    
    model = Perceptron(**params)
    # Using 3-fold cross-validation to target max Recall
    score = cross_val_score(model, X_train_scaled, y_train, cv=3, scoring='recall', n_jobs=-1).mean()
    return score

with mlflow.start_run(run_name="Perceptron_Optuna"):
    study = optuna.create_study(direction="maximize")
    start_time = time.time()
    study.optimize(objective, n_trials=15)
    duration = time.time() - start_time
    
    # Retrain final optimal model
    best_perceptron_opt = Perceptron(**study.best_params, random_state=42)
    best_perceptron_opt.fit(X_train_scaled, y_train)
    
    mlflow.log_params(study.best_params)
    mlflow.log_param("optimization", "optuna")
    log_metrics(y_test, best_perceptron_opt.predict(X_test_scaled), duration)

[I 2026-05-18 17:04:28,257] A new study created in memory with name: no-name-02bfe793-6b36-4e35-a4ed-d444a1f5d3ce
[I 2026-05-18 17:04:28,724] Trial 0 finished with value: 0.926853137956539 and parameters: {'eta0': 0.029018174590557635, 'penalty': 'l1', 'alpha': 0.025813453234052867, 'max_iter': 1172}. Best is trial 0 with value: 0.926853137956539.
[I 2026-05-18 17:04:29,155] Trial 1 finished with value: 0.8583894540283769 and parameters: {'eta0': 0.017182266336743884, 'penalty': None, 'alpha': 3.178746294164261e-05, 'max_iter': 1085}. Best is trial 0 with value: 0.926853137956539.
[I 2026-05-18 17:04:29,659] Trial 2 finished with value: 0.7962219123070192 and parameters: {'eta0': 0.09577867875812116, 'penalty': 'l1', 'alpha': 0.008503755704186744, 'max_iter': 1609}. Best is trial 0 with value: 0.926853137956539.
[I 2026-05-18 17:04:30,090] Trial 3 finished with value: 0.8528475373460842 and parameters: {'eta0': 0.006467628448493787, 'penalty': 'l2', 'alpha': 0.007743244807365256, 'max_

## Runs Summary (Common Hyperparameters and Metrics)

| Run | Optimization | alpha | eta0 | penalty | max_iter | random_state | Accuracy | F1 | Recall |
|---|---|---:|---:|---|---:|---:|---:|---:|---:|
| Perceptron_Baseline_Defaults | none_default | 0.0001 | 1.0 | None | 1000 | 42 | 0.8202 | 0.8429694323 | 0.8043333333 |
| Perceptron_GridSearch | GridSearchCV | 0.01 | 0.0001 | l1 | 1000 | 42 | 0.8937 | 0.9028069855 | 0.8228333333 |
| Perceptron_Optuna | optuna | 0.0001559297 | 0.0005921161 | l1 | 1925 | 42 | 0.8937 | 0.9028069855 | 0.8228333333 |

Notes:
- Only the hyperparameters common to the three runs are shown here.
- Training time is intentionally excluded from decision-making for Streamlit inference.

## Best Run Justification for Streamlit

For Streamlit deployment, the best run is **Perceptron_GridSearch**.

Why this run:
- It ties for the best predictive balance across the three key metrics (Accuracy = **0.8937**, F1 = **0.9028**, Recall = **0.8228**), matching Optuna and clearly outperforming the baseline.
- It is easier to explain and reproduce in production (`eta0=0.0001`, `penalty=l1`, `alpha=0.01`) without the extra optimization complexity of Optuna. **Perceptron_GridSearch** is the most practical choice.


In [6]:
from sklearn.metrics import accuracy_score, f1_score, recall_score

def summarize_split(y_true, y_pred):
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
    }

diagnostic_rows = [
    {
        'run': 'Perceptron_Baseline_Defaults',
        'split': 'train',
        **summarize_split(y_train, perceptron_base.predict(X_train_scaled))
    },
    {
        'run': 'Perceptron_Baseline_Defaults',
        'split': 'test',
        **summarize_split(y_test, perceptron_base.predict(X_test_scaled))
    },
    {
        'run': 'Perceptron_GridSearch',
        'split': 'train',
        **summarize_split(y_train, best_perceptron.predict(X_train_scaled))
    },
    {
        'run': 'Perceptron_GridSearch',
        'split': 'test',
        **summarize_split(y_test, y_pred_grid)
    },
    {
        'run': 'Perceptron_Optuna',
        'split': 'train',
        **summarize_split(y_train, best_perceptron_opt.predict(X_train_scaled))
    },
    {
        'run': 'Perceptron_Optuna',
        'split': 'test',
        **summarize_split(y_test, best_perceptron_opt.predict(X_test_scaled))
    },
]

pd.DataFrame(diagnostic_rows)

,run,split,accuracy,f1,recall
0,Perceptron_Baseline_Defaults,train,0.821163,0.843945,0.805992
1,Perceptron_Baseline_Defaults,test,0.820200,0.842969,0.804333
2,Perceptron_GridSearch,train,0.896350,0.905455,0.827243
3,Perceptron_GridSearch,test,0.893700,0.902807,0.822833
4,Perceptron_Optuna,train,0.896350,0.905455,0.827243
5,Perceptron_Optuna,test,0.893700,0.902807,0.822833
